# Uspokajanie się wskazań po odsypie

Jak długo po końcu sygnału odsypu wskazania wagi silosu są niespokojne (drgania po odsypie, wibrator) i czy wskazanie dalej się zmienia. Od tego zależy, skąd brać wskazanie „po” w metodach różnicy (obecnie mediana z 3 s po końcu albo najbliższa próbka). Zbiór: baza `2026-10-18 obiekt`, sesja 9, wariant `CURRENT_VARIANT`; obliczenia w `dms_analysis.settling`.

Kroki:

1. **Odsypy** przyjęte, po których przez co najmniej `AFTER_S` nie zaczyna się żadne inne zdarzenie (w seriach odsypów przerwy są zwykle krótsze, więc to część odsypów).
2. **Chwile kontrolne** — losowe chwile we fragmentach bez zdarzeń (`dms_analysis.between_events_fragments`), po `CONTROL_PER_DISPENSE` na odsyp, z tej samej godziny co koniec odsypu (zmienność wskazań zależy od pory dnia), ziarno `CONTROL_SEED`. Pokazują, jak zachowują się wskazania, kiedy nic się nie dzieje.
3. **Zmiana z sekundy na sekundę** — średnia bezwzględna zmiana wskazania w kolejnych sekundach od końca odsypu, od początku odsypu (dla porównania: drgania w czasie odsypu) i od chwili kontrolnej.
4. **Czas uspokojenia** każdego odsypu — pierwsza sekunda po końcu, od której średnia |zmiana| w oknie `STEADY_WINDOW_S` nie przekracza progu spokoju = `STEADY_QUANTILE` takich okien w chwilach kontrolnych.
5. **Poziom wskazania** — odchyłka od mediany z ostatnich `REFERENCE_S` okna: czy po końcu odsypu wskazanie systematycznie dalej spada albo wraca, w porównaniu z chwilami kontrolnymi.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from IPython.display import display

from dms_analysis.between_events_fragments import prepare_samples
from dms_analysis.prepared import load_session
from dms_analysis.settling import (
    aligned,
    control_moments,
    deviation_bands,
    deviation_figure,
    free_dispenses,
    overlay_figure,
    settling_times,
    steady_threshold,
    step_figure,
    step_profile,
)
from dms_analysis.signals import CURRENT_VARIANT
from dms_analysis.snapshot import ANALYSIS_SESSION, ANALYSIS_SOURCE
from dms_analysis.tables import collapsible

DB_PATH = Path("../data/prepared/obiekt-sesja-9.db")
FIGURE_WIDTH = 1200

In [ ]:
session = load_session(DB_PATH, ANALYSIS_SOURCE, ANALYSIS_SESSION, CURRENT_VARIANT)
samples = prepare_samples(session)

dispenses = free_dispenses(session)
controls = control_moments(samples, dispenses)

after_end = aligned(session, dispenses["end"])
after_start = aligned(session, dispenses["start"])
control = aligned(session, controls)

print(f"odsypów z wolnym czasem po końcu: {len(dispenses)}, chwil kontrolnych: {len(controls)}")

## Zmiana z sekundy na sekundę

Linia przerywana: próg spokoju z chwil kontrolnych.

In [ ]:
profile = step_profile(after_end, after_start, control)
threshold = steady_threshold(control)

print(f"próg spokoju: {threshold:.1f} kg")
step_figure(profile, threshold, FIGURE_WIDTH).show()

## Czas uspokojenia

Rozkład i lista odsypów od najdłuższego uspokajania (brak wartości = nie uspokoił się w oknie).

In [ ]:
settling = settling_times(after_end, dispenses, threshold)

display(settling["settled_s"].describe(percentiles=[0.5, 0.75, 0.9, 0.95]).round(1))
display(collapsible(settling.sort_values("settled_s", ascending=False), "Czas uspokojenia odsypów"))

## Poziom wskazania po końcu odsypu

Pasmo 10.–90. percentyla odchyłki od mediany z końca okna, po końcu odsypu i w chwilach kontrolnych; pod spodem wskazania po każdym odsypie osobno (najechanie pokazuje odsyp).

In [ ]:
display(collapsible(deviation_bands(after_end).round(0), "Odchyłka po końcu odsypu"))

deviation_figure(after_end, control, FIGURE_WIDTH).show()
overlay_figure(after_end, FIGURE_WIDTH).show()